In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim.lr_scheduler as lr_scheduler
from comet_ml import Experiment
from scipy import linalg
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from src.utils.samplers.data import DatasetSampler, PairedLoaderSampler
from src.utils.samplers.synthetic import StandardNormalSampler, SwissRollSampler
from src.utils.training.weather_notebook import (
    load_weather_tensors,
    paired_sampler_weather,
    unpaired_sampler_weather,
)

from src.utils.notebook_setup import ensure_repo_imports

REPO_ROOT = ensure_repo_imports()

tsne = TSNE(n_components=2, random_state=50)


# Data preparation

### PS
1) Source  
$X \in \mathbb{R}^{N \times d_{1}}, N - \text{number of locations}, d_{1} - \text{features dim}$ \
$x = (\mu, \sigma) - \text{for a given location in June}$ \
$N = 1396, d_{1} = 188$ 

2) $Y \in \mathbb{R}^{N \times M \times d_{2}}, N - \text{number of locations}, M - \text{measurements for a given location in January by day}$ \
$M = [1, 31], d_{2} = 94$ 

In [2]:
##########################################
#-------------- RAW DATA -----------------
##########################################

import numpy as np
import pandas as pd

root = '../tabred/kal/weather'

data = np.load(f'{root}/X_num.npy')
data = np.stack([d for d in data if sum(np.isnan(d)) == 0])
data_csv = pd.read_csv(f'{root}/csv/X_num.csv')
#train_data = data[train_idx]
#test_data = data[test_idx]

target = np.load(f'{root}/Y.npy')
meta = np.load(f'{root}/X_meta.npy')
meta = np.stack([meta[i] for i, d in enumerate(data) if sum(np.isnan(d)) == 0])
meta_csv = pd.read_csv(f'{root}/csv/X_meta.csv')

names = list(data_csv.columns)
names.append('location')
data_new = np.concatenate((data, meta[:, -2].reshape(-1, 1)), axis=1)

In [3]:
#########################################################
#--------------- Month/location splitted ---------------- 
#########################################################
scaler = StandardScaler()

dict_location_src = {}
for d in data_new:
    if d[-2] == 1.0:
        d_new = d[:-7]
        try:
            dict_location_src[d[-1]].append(d_new)
        except KeyError:
            dict_location_src[d[-1]] = []
            dict_location_src[d[-1]].append(d_new)
     

dict_location_src_new = {}
for key in dict_location_src.keys():
    item = dict_location_src[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_src_new[key] = item
dict_location_src = dict_location_src_new
# ------------------------------------------------------------

dict_location_trg = {}
for d in data_new:
    if d[-2] == 6.0:
        d_new = d[:-7]
        try:
            dict_location_trg[d[-1]].append(d_new)
        except KeyError:
            dict_location_trg[d[-1]] = []
            dict_location_trg[d[-1]].append(d_new)
    
dict_location_trg_new = {}
for key in dict_location_trg.keys():
    item = dict_location_trg[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_trg_new[key] = item
dict_location_trg = dict_location_trg_new

print(len(dict_location_trg), len(dict_location_src))

1653 1578


In [4]:
#########################################################
#--------------------- X, Y paired ----------------------
#########################################################

chosen_locs = list(dict_location_trg.keys())[:200]
X_pair_orig, Y_pair_orig = [], []
for key in dict_location_src.keys():
    if key not in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_pair_orig.append(x)
    item_trg = dict_location_trg[key]
    Y_pair_orig.append(item_trg) # sample
X_pair_orig = np.stack(X_pair_orig)


#########################################################
#----------------------- X, Y ---------------------------
#########################################################

# N x 1 x 2D - src
# N x M x D - trg
 
# sampling: 
# b x 1 x 2D,
# b x M x D -> sample -> b x 1 x D

X_orig = []
for key in dict_location_src.keys():
    if key in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_orig.append(x)
X_orig = np.stack(X_orig)

Y_orig = []
for key in dict_location_trg.keys():
    if key in chosen_locs:
        continue
    item_trg = dict_location_trg[key]
    Y_orig.append(item_trg) # sample

In [5]:
print(X_orig.shape, len(Y_orig), X_pair_orig.shape, len(Y_pair_orig))

(1386, 188) 1453 (192, 188) 192


# Running

In [24]:
source_data = X_orig
target_data = Y_orig[0]
X_DIM = source_data.shape[1]
Y_DIM = target_data.shape[1]
#X_DIM = data_set["features"].shape[1]
#Y_DIM = data_set["features"].shape[1]
assert X_DIM > 1
assert Y_DIM > 1

OUTPUT_SEED = 50

N_POTENTIALS = 10
M_POTENTIALS = 1 #10
EPSILON = 1
A_DIAGONAL_INIT = 0.5
L_PAIRED_SAMPLES = len(X_pair_orig)
M_X_UNPAIRED_SAMPLES = 0
N_Y_UNPAIRED_SAMPLES = 0

BATCH_SIZE = 128
SAMPLING_BATCH_SIZE = 128

D_LR = 3e-4  # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_GRADIENT_MAX_NORM = float("inf")

NUM_LABELED = 10
TRAIN_SUBSET_SIZE = 2

PLOT_EVERY = 1000
MAX_STEPS = 20000
CONTINUE = -1

In [25]:
EXP_COST = "MLP_deep_deep"
EXP_COST_INCLUDED = True
EXP_META_INFO = ""
EXP_NAME = (
    f"EBiEOT-GMM_Batch_Effect_"
    + f"EPSILON_{EPSILON}_"
    + f"N_{N_POTENTIALS}_"
    + f"M_{M_POTENTIALS}_"
    + f"with_{EXP_COST}_"
    + f"cost_included_{EXP_COST_INCLUDED}_"
    + f"N_PAIRED_{NUM_LABELED}_"
    + f"M_UNPAIRED_{len(source_data)}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR=D_LR,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    N_PAIRED_SAMPLES=NUM_LABELED,
    M_UNPAIRED_SAMPLES=len(source_data),
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)


In [26]:
#pytorch_total_params = sum(p.numel() for p in D.parameters())
#pytorch_total_params

## Ablation Study

In [27]:
def paired_sampler(X_pair, Y_pair, b_size):
    idxs = np.random.randint(low=0, high=len(X_pair)-1, size=b_size)
    x_pair_batch = torch.tensor(X_pair[idxs]).to('cuda')
    y_pair_batch = np.stack([Y_pair[idx][random.randint(0, len(Y_pair[idx])-1)] for idx in idxs])
    y_pair_batch = torch.tensor(y_pair_batch).to('cuda')
    return x_pair_batch.to(torch.float32), y_pair_batch.to(torch.float32)

def unpaired_sampler(X, Y, b_size):
    # UNPAIRED SAMPLER
    idxs = np.random.randint(low=0, high=len(X)-1, size=b_size)
    idxs_y = np.array([len(X) - idx - 1 for idx in idxs])

    x_batch = torch.tensor(X[idxs]).to('cuda')
    y_batch = np.stack([Y[idx][random.randint(0, len(Y[idx])-1)] for idx in idxs_y])
    y_batch = torch.tensor(y_batch).to('cuda')
    return x_batch.to(torch.float32), y_batch.to(torch.float32)

In [28]:
from src.utils.samplers.discrete_ot import OTPlanSampler
from src.utils.datasets.paired import generate_paired_data, get_GT_points, get_paired_sampler
import torch.nn.functional as F
from scipy import linalg


In [29]:
nz = 10
from src.networks.gan.generator import MLPGenerator
from src.networks.gan.descriminator import MLPDiscriminator

netG = MLPGenerator(
        x_dim=X_DIM,
        out_dim=Y_DIM,
        z_dim=nz,
        layers=[256, 256, 256],
    ).to('cuda')

netD = MLPDiscriminator(x_dim=Y_DIM,
                        layers=[256, 256, 256]).to('cuda')
optimizerD = torch.optim.Adam(netD.parameters())
optimizerG = torch.optim.Adam(netG.parameters())

schedulerG = torch.optim.lr_scheduler.CosineAnnealingLR(optimizerG, 5000, eta_min=1e-5)
schedulerD = torch.optim.lr_scheduler.CosineAnnealingLR(optimizerD, 5000, eta_min=1e-5)


In [30]:
history = {
        "D_loss": [],
        "G_loss": [],
    }

In [31]:
MAX_STEPS = 10000
from torch.distributions.normal import Normal 
experiment = Experiment(project_name="inverse_ot")
experiment.set_name(EXP_NAME)
stats = []
D_loss = []
fids = []
fids2 = []
device = 'cuda'

# Splitting
L_PAIRED_SAMPLES = 90
L_UNPAIRED_SAMPLES = 500
X, Y = X_orig[:L_UNPAIRED_SAMPLES], Y_orig[-L_UNPAIRED_SAMPLES:]
X_pair, Y_pair = X_pair_orig[:L_PAIRED_SAMPLES], Y_pair_orig[:L_PAIRED_SAMPLES]
X_pair_test, Y_pair_test = X_pair_orig[-100:], Y_pair_orig[-100:]


for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):
        #########################
        # Discriminator training
        #########################
        for p in netD.parameters():
            p.requires_grad = True

        netD.zero_grad()

        ###################################
        # Sample real data
        X_unpaired, Y_unpaired = unpaired_sampler(X, Y, BATCH_SIZE)
        Y_unpaired.requires_grad = True

        ###################################
        # Optimizing loss on real data
        D_real = netD(Y_unpaired)

        errD_real = F.softplus(-D_real)
        errD_real = errD_real.mean()

        errD_real.backward(retain_graph=True)

        ###################################
        # R_1(\phi) regularization
        if 1 is None or step % 1 == 0:
            grad_real = torch.autograd.grad(
                outputs=D_real.sum(),
                inputs=Y_unpaired,
                create_graph=True,
            )[0]
            grad_penalty = (grad_real.view(grad_real.size(0), -1).norm(2, dim=1) ** 2).mean()

            grad_penalty = 0.01 / 2 * grad_penalty
            grad_penalty.backward()

        ###################################
        # Sample vector from latent space for generation
        latent_z = torch.randn(BATCH_SIZE, nz, device=device)

        ###################################
        # Sample fake output
        x_0_predict = netG(X_unpaired.detach(), latent_z)

        ###################################
        # Optimize loss on fake data
        output = netD(x_0_predict).view(-1)

        errD_fake = F.softplus(output)
        errD_fake = errD_fake.mean()
        errD_fake.backward()

        errD = errD_real + errD_fake

        D_loss.append(errD.item())
        #print(f'D Loss {errD.item()}')

        ###################################
        # Update weights of netD
        optimizerD.step()

        #############################################################

        #########################
        # Generator training
        #########################
        for p in netD.parameters():
            p.requires_grad = False
        netG.zero_grad()
        
        ###################################
        # Sample pairs for training
        unp_sample, _ = unpaired_sampler(X, Y, BATCH_SIZE)
        X_paired, Y_paired = paired_sampler(X_pair, Y_pair, BATCH_SIZE)

        ###################################
        # Sample vector from latent space for generation
        latent_z = torch.randn(BATCH_SIZE, nz, device=device)
        latent_z0 = torch.randn(BATCH_SIZE, nz, device=device)

        ###################################
        # Sample fake output
        x_paired_predict = netG(X_paired.detach(), latent_z)
        x_unp_predict = netG(unp_sample.detach(), latent_z0)

        ###################################
        # Optimize loss on fake data
        output = netD(x_unp_predict).view(-1)

        ###################################
        # Update weights of netG
        errG = F.softplus(-output)
        errG_mse = F.mse_loss(Y_paired, x_paired_predict)
        errG = (errG + errG_mse).mean()

        errG.backward()
        optimizerG.step()

        #print(f'G Loss {errG.item()}')
        # LR-Scheduling step
        schedulerG.step()
        schedulerD.step()
        
        with torch.no_grad():
            if step % 1000 == 0:
                for x, y in zip(X_pair_test, Y_pair_test):
                    x = torch.tensor(x).unsqueeze(0).to(device)
                    y = torch.tensor(y).to(device)
                    samples = []
                    for _ in range(len(y)):
                        latent_z = torch.randn(len(x), nz, device=device)
                        t = torch.randint(0, 1, (latent_z.size(0),), device=device)
                        sample = netG(x.detach().float(), latent_z)
                        samples.append(sample.cpu())
                    
                    try:
                        samples = torch.cat(samples, dim=0).cpu()
                        loc = torch.tensor(samples.cpu()).mean(dim=0)
                        scale = torch.tensor(samples.cpu()).std(dim=0)
                        loss = -Normal(loc, scale).log_prob(y.cpu()).sum()
                        fids.append(loss)
                    except ValueError:
                        continue
                    
                    """
                    fid_samples = np.array(torch.cat(samples, dim=0).cpu())
                    fid_samples_2 = np.array(y.cpu())

                    mu1 = np.mean(fid_samples, axis=0)
                    sigma1 = np.cov(fid_samples, rowvar=False)
                    mu2 = np.mean(fid_samples_2, axis=0)
                    sigma2 = np.cov(fid_samples_2, rowvar=False)

                    diff = mu1 - mu2
                    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
                    tr_covmean = np.trace(covmean)
                    fid = (diff.dot(diff) + np.trace(sigma1) +  np.trace(sigma2) - 2 * tr_covmean)
                    fids.append(fid.real)
                    fids2.append(fid.real / np.var(fid_samples_2))
                    """
                    
                print(np.mean(fids))

<>:42: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:42: SyntaxWarning: "is" with a literal. Did you mean "=="?
/var/tmp/ipykernel_53469/2634283775.py:42: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if 1 is None or step % 1 == 0:
  0%|                                                 | 0/10000 [00:00<?, ?it/s]/var/tmp/ipykernel_53469/2634283775.py:134: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loc = torch.tensor(samples.cpu()).mean(dim=0)
/var/tmp/ipykernel_53469/2634283775.py:135: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(samples.cpu()).std(dim=0)
  0%|                                        | 11/10000 [00:01<13:44, 12.12it/s]

8634169.508643506


 10%|███▊                                  | 1013/10000 [00:13<05:06, 29.30it/s]

5600019.874016167


 20%|███████▋                              | 2016/10000 [00:25<04:35, 28.97it/s]

...

 80%|██████████████████████████████▍       | 8008/10000 [01:39<01:35, 20.87it/s]

4327468.715367576


 90%|██████████████████████████████████▏   | 9012/10000 [01:51<00:34, 28.74it/s]

4823665.808403021


100%|█████████████████████████████████████| 10000/10000 [02:02<00:00, 81.41it/s]


In [32]:
a = [-3574, -9544, -4823]
np.mean(a), np.std(a)

(-5980.333333333333, 2570.964842665536)